# 专用工装价格到期剥离

供应商定点会议确定工装分摊数量与价格 → MRP 累计收货 → 检查是否达到剥离条件。全量扫描。


In [ ]:
import pandas as pd
from datetime import datetime
import os

TODAY = datetime.now().strftime("%Y%m%d")
OUTPUT_DIR = "../output/专用工装到期剥离"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, f"to_remind_{TODAY}.xlsx")
print(f"运行日期: {TODAY}  模式: 全量扫描")

## Step 1：读取工装分摊台账

In [ ]:
df_amort = pd.read_excel("../data/定点会_工装分摊台账.xlsx")
df_amort["mk"] = df_amort["零件号"].astype(str)+'_'+df_amort["供应商编码"].astype(str)
print(f"工装台账: {len(df_amort)} parts")

## Step 2：MRP 累计收货汇总

In [ ]:
df_mrp = pd.read_excel("../data/MRP_收货记录表.xlsx")
df_mrp["mk"] = df_mrp["零件号"].astype(str)+'_'+df_mrp["供应商编码"].astype(str)
df_cumsum = df_mrp.groupby("mk").agg(累计收货量=("收货数量","sum"), 最近收货日期=("收货日期","max")).reset_index()
print(f"MRP groups with records: {len(df_cumsum)}")

## Step 3：EPS 价格

In [ ]:
df_eps = pd.read_excel("../data/eps3_mass.xlsx")
df_eps = df_eps[df_eps["状态"].isin(["已发布","已确认","已审批"])].copy()
df_eps["mk"] = df_eps["零件号"].astype(str)+'_'+df_eps["供应商编码"].astype(str)
df_eps["创建时间"] = pd.to_datetime(df_eps["创建时间"], errors="coerce")
df_eps_l = df_eps.sort_values(["mk","创建时间"], ascending=[True,False]).groupby("mk", as_index=False).first()
print(f"EPS: {len(df_eps_l)}")

## Step 4：三源匹配筛选

In [ ]:
m = df_amort.merge(df_cumsum[["mk","累计收货量","最近收货日期"]], on="mk", how="left")
m["累计收货量"] = m["累计收货量"].fillna(0).astype(int)
m["达标"] = m["累计收货量"] >= m["分摊数量"]
print(f"应剥离: {m['达标'].sum()} / {len(m)}")

should = m[m["达标"]].copy()
if len(should)==0:
    print("无达标项"); exit()

r = should.merge(df_eps_l[["mk","零件裸价"]], on="mk", how="left")
r.rename(columns={"零件裸价":"当前EPS裸价"}, inplace=True)

def status(row):
    if pd.isna(row["当前EPS裸价"]): return "待核实"
    if abs(row["当前EPS裸价"]-row["剥离后结算价"])<0.01: return "已剥离"
    return "待剥离"
r["价格状态"] = r.apply(status, axis=1)
df_remind = r[r["价格状态"]!="已剥离"]
print(f"待处理: {len(df_remind)}  {df_remind['价格状态'].value_counts().to_dict()}")

## Step 5：导出

In [ ]:
if len(df_remind)>0:
    df_remind["分摊总金额"] = df_remind["分摊数量"]*df_remind["分摊价格(元)"]
    df_remind["超标比例"] = ((df_remind["累计收货量"]-df_remind["分摊数量"])/df_remind["分摊数量"]).round(4)
    df_remind["检查日期"] = datetime.now().strftime("%Y/%m/%d")
    cols = ["零件号","零件名称","供应商编码","供应商名称","专用工装名称","分摊数量","累计收货量","超标比例","分摊价格(元)","分摊总金额","当前结算价(含分摊)","剥离后结算价","当前EPS裸价","价格状态","定点会议号","定点会议日期","采购工程师","最近收货日期","检查日期"]
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    df_remind[cols].to_excel(OUTPUT_FILE, index=False)
    print(f"Exported: {OUTPUT_FILE} ({len(df_remind)} rows)")
else:
    print("所有达标零件已剥离")